In [ ]:
!pip install yfinance prophet streamlit pyngrok vaderSentiment transformers torch newsapi-python praw plotly pandas numpy scikit-learn tensorflow keras ta --quiet


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 83.3 MB/s eta 0:00:00


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from prophet import Prophet
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


In [ ]:
def fetch_stock_data(ticker: str, period: str = "2y") -> pd.DataFrame:
    """
    Fetch historical stock data from Yahoo Finance.
    ticker: e.g. 'AAPL', 'TSLA', 'MSFT'
    period: '1y', '2y', '5y'
    """
    stock = yf.Ticker(ticker)
    df = stock.history(period=period)
    df.reset_index(inplace=True)
    df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
    df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
    print(f"✅ Fetched {len(df)} rows for {ticker}")
    return df

TICKER = "AAPL"   # ← Change this to any stock ticker you want
df_stock = fetch_stock_data(TICKER, period="2y")
df_stock.tail()


✅ Fetched 502 rows for AAPL


,Date,Open,High,Low,Close,Volume
497,2026-04-23,275.049988,275.769989,271.649994,273.429993,33399600
498,2026-04-24,272.760010,273.059998,269.649994,271.059998,38157100
499,2026-04-27,266.089996,268.359985,265.070007,267.609985,41466800
500,2026-04-28,272.339996,273.230011,268.660004,270.709991,39978600
501,2026-04-29,267.549988,271.040009,267.040009,270.320007,12455082


In [ ]:
def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Moving Averages
    df['MA_7']  = df['Close'].rolling(window=7).mean()
    df['MA_21'] = df['Close'].rolling(window=21).mean()
    df['MA_50'] = df['Close'].rolling(window=50).mean()

    # RSI
    delta = df['Close'].diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = df['Close'].ewm(span=12).mean()
    ema26 = df['Close'].ewm(span=26).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_Signal'] = df['MACD'].ewm(span=9).mean()

    # Bollinger Bands
    sma20 = df['Close'].rolling(20).mean()
    std20 = df['Close'].rolling(20).std()
    df['BB_Upper'] = sma20 + 2 * std20
    df['BB_Lower'] = sma20 - 2 * std20
    df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']

    # Daily Return & Volatility
    df['Daily_Return'] = df['Close'].pct_change()
    df['Volatility']   = df['Daily_Return'].rolling(10).std()

    df.dropna(inplace=True)
    print(f"✅ Technical indicators added. Shape: {df.shape}")
    return df

df_stock = add_technical_indicators(df_stock)
df_stock.tail()


✅ Technical indicators added. Shape: (453, 17)


,Date,Open,High,Low,Close,Volume,MA_7,MA_21,MA_50,RSI,MACD,MACD_Signal,BB_Upper,BB_Lower,BB_Width,Daily_Return,Volatility
497,2026-04-23,275.049988,275.769989,271.649994,273.429993,33399600,269.411429,260.115237,260.234200,67.043043,3.919685,2.108866,276.326243,244.653756,31.672488,0.000952,0.017876
498,2026-04-24,272.760010,273.059998,269.649994,271.059998,38157100,270.072859,260.993333,260.145400,62.007903,4.019427,2.490978,277.481795,245.315204,32.166591,-0.008668,0.018349
499,2026-04-27,266.089996,268.359985,265.070007,267.609985,41466800,270.674286,261.694284,260.262999,64.430353,3.776553,2.748093,277.493387,247.184610,30.308776,-0.012728,0.018934
500,2026-04-28,272.339996,273.230011,268.660004,270.709991,39978600,270.742855,262.737617,260.561599,62.674400,3.790524,2.956579,277.194081,249.891915,27.302166,0.011584,0.019016
501,2026-04-29,267.549988,271.040009,267.040009,270.320007,12455082,270.352857,263.865712,260.690399,60.828406,3.727162,3.110696,277.526982,251.212015,26.314966,-0.001441,0.016952


In [ ]:
analyzer = SentimentIntensityAnalyzer()

# --- Option A: Simulated headlines (works without any API key) ---
def generate_mock_sentiment(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """
    Generate mock sentiment scores for each trading day.
    In production, replace this with real NewsAPI or Reddit data.
    """
    np.random.seed(42)
    sentiment_scores = []

    for i, row in df.iterrows():
        # Simulate sentiment that loosely correlates with price movement
        base = row['Daily_Return'] if 'Daily_Return' in df.columns else 0
        noise = np.random.normal(0, 0.2)
        score = np.clip(base * 3 + noise, -1, 1)
        sentiment_scores.append({
            'Date': row['Date'],
            'sentiment_compound': round(score, 4),
            'sentiment_pos': max(0, score),
            'sentiment_neg': max(0, -score),
        })

    sentiment_df = pd.DataFrame(sentiment_scores)
    print(f"✅ Sentiment generated for {len(sentiment_df)} days")
    return sentiment_df

df_sentiment = generate_mock_sentiment(df_stock, TICKER)
df_sentiment.head()


✅ Sentiment generated for 453 days


,Date,sentiment_compound,sentiment_pos,sentiment_neg
0,2024-07-10,0.1558,0.155754,0.000000
1,2024-07-11,-0.0973,0.000000,0.097315
2,2024-07-12,0.1687,0.168690,0.000000
3,2024-07-15,0.3548,0.354836,0.000000
4,2024-07-16,-0.0415,0.000000,0.041455


In [ ]:
# --- Option B: Real headlines from NewsAPI ---
# Sign up free at https://newsapi.org/ to get an API key

NEWS_API_KEY = "ce65b7b881b74b53868aa4963024fd72"   # ← Paste your key here

def fetch_real_sentiment(ticker: str, api_key: str, days_back: int = 30) -> pd.DataFrame:
    try:
        from newsapi import NewsApiClient
        newsapi = NewsApiClient(api_key=api_key)

        company_names = {
            "AAPL": "Apple", "TSLA": "Tesla", "MSFT": "Microsoft",
            "GOOGL": "Google", "AMZN": "Amazon", "NVDA": "NVIDIA"
        }
        query = company_names.get(ticker, ticker)

        from_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')
        articles = newsapi.get_everything(q=query, from_param=from_date,
                                          language='en', sort_by='publishedAt',
                                          page_size=100)

        records = []
        for a in articles.get('articles', []):
            headline = (a.get('title') or '') + ' ' + (a.get('description') or '')
            score    = analyzer.polarity_scores(headline)
            date     = pd.to_datetime(a['publishedAt']).normalize()
            records.append({'Date': date, **score})

        df = pd.DataFrame(records)
        if df.empty:
            print("⚠️ No articles found, using mock sentiment.")
            return None

        df = df.groupby('Date').mean().reset_index()
        df.rename(columns={'compound': 'sentiment_compound',
                            'pos': 'sentiment_pos',
                            'neg': 'sentiment_neg'}, inplace=True)
        print(f"✅ Real sentiment fetched: {len(df)} days")
        return df

    except Exception as e:
        print(f"⚠️ NewsAPI error: {e}. Falling back to mock sentiment.")
        return None

# Uncomment to use real data:
real_sent = fetch_real_sentiment(TICKER, NEWS_API_KEY)
df_sentiment = real_sent if real_sent is not None else df_sentiment


✅ Real sentiment fetched: 1 days


In [ ]:
def merge_data(df_stock, df_sent):
    df_s = df_stock.copy()
    df_n = df_sent.copy()

    # Strip timezone info from both Date columns
    df_s['Date'] = pd.to_datetime(df_s['Date']).dt.tz_localize(None)
    df_n['Date'] = pd.to_datetime(df_n['Date']).dt.tz_localize(None)

    df = pd.merge(df_s, df_n, on='Date', how='left')
    df['sentiment'].fillna(0, inplace=True)
    df.sort_values('Date', inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"✅ Merged shape: {df.shape}")
    return df

df = merge_data(df_stock, df_sent)
df.tail()


✅ Merged shape: (453, 18)


,Date,Open,High,Low,Close,Volume,MA_7,MA_21,MA_50,RSI,MACD,MACD_Signal,BB_Upper,BB_Lower,BB_Width,Daily_Return,Volatility,sentiment
448,2026-04-23,275.049988,275.769989,271.649994,273.429993,33399600,269.411429,260.115237,260.234200,67.043043,3.919685,2.108866,276.326243,244.653756,31.672488,0.000952,0.017876,-0.0999
449,2026-04-24,272.760010,273.059998,269.649994,271.059998,38157100,270.072859,260.993333,260.145400,62.007903,4.019427,2.490978,277.481795,245.315204,32.166591,-0.008668,0.018349,-0.2378
450,2026-04-27,266.089996,268.359985,265.070007,267.609985,41466800,270.674286,261.694284,260.262999,64.430353,3.776553,2.748093,277.493387,247.184610,30.308776,-0.012728,0.018934,-0.0507
451,2026-04-28,272.339996,273.230011,268.660004,270.709991,39978600,270.742855,262.737617,260.561599,62.674400,3.790524,2.956579,277.194081,249.891915,27.302166,0.011584,0.019016,0.2258
452,2026-04-29,267.549988,271.040009,267.040009,270.320007,12455082,270.352857,263.865712,260.690399,60.828406,3.727162,3.110696,277.526982,251.212015,26.314966,-0.001441,0.016952,-0.2015


In [ ]:
# Updated FEATURES to match the 'sentiment' column name
FEATURES = ['Close', 'Volume', 'MA_7', 'MA_21', 'RSI', 'MACD',
            'BB_Width', 'Volatility', 'sentiment']
SEQ_LEN  = 30

def prepare_lstm_data(df, features, seq_len=30, split=0.8):
    # Verify all features exist before proceeding
    missing = [f for f in features if f not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}\nAvailable: {list(df.columns)}")

    scaler     = MinMaxScaler()
    scaled     = scaler.fit_transform(df[features].values)
    target_idx = features.index('Close')

    X, y = [], []
    for i in range(seq_len, len(scaled)):
        X.append(scaled[i - seq_len:i])
        y.append(scaled[i, target_idx])

    X, y      = np.array(X), np.array(y)
    split_idx = int(len(X) * split)
    return (X[:split_idx], X[split_idx:],
            y[:split_idx], y[split_idx:],
            scaler, target_idx)

X_train, X_test, y_train, y_test, scaler, t_idx = prepare_lstm_data(df, FEATURES)
print(f"✅ Train: {X_train.shape}  |  Test: {X_test.shape}")


✅ Train: (338, 30, 9)  |  Test: (85, 30, 9)


In [ ]:
def build_lstm_model(seq_len: int, n_features: int) -> Sequential:
    model = Sequential([
        Bidirectional(LSTM(128, return_sequences=True),
                      input_shape=(seq_len, n_features)),
        Dropout(0.2),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='huber', metrics=['mae'])
    model.summary()
    return model

model = build_lstm_model(SEQ_LEN, len(FEATURES))

early_stop = EarlyStopping(monitor='val_loss', patience=10,
                           restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
print("✅ Training complete!")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 30, 256)        │       141,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 30, 64)         │        82,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 236,449 (923.63 KB)

 Trainable params: 236,449 (923.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 80ms/step - loss: 0.0279 - mae: 0.1828 - val_loss: 0.0819 - val_mae: 0.4026
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0077 - mae: 0.0997 - val_loss: 0.0430 - val_mae: 0.2906
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0067 - mae: 0.0891 - val_loss: 0.0103 - val_mae: 0.1384
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0056 - mae: 0.0819 - val_loss: 0.0124 - val_mae: 0.1527
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0046 - mae: 0.0734 - val_loss: 0.0158 - val_mae: 0.1734
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0051 - mae: 0.0786 - val_loss: 0.0258 - val_mae: 0.2241
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0043 - mae: 0.0725 - val_loss: 0.0065 - val_mae: 0.1069
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0050 - mae: 0.0795 - val_loss: 0.0103 - val_mae: 0.1390
Epoch 9/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - lo

In [ ]:
def inv_scale(arr, scaler, t_idx, n_feat):
    dummy = np.zeros((len(arr), n_feat))
    dummy[:, t_idx] = arr.flatten()
    return scaler.inverse_transform(dummy)[:, t_idx]

y_pred = model.predict(X_test).flatten()

# Use t_idx (returned from Cell 8)
pred   = inv_scale(y_pred,  scaler, t_idx, len(FEATURES))
actual = inv_scale(y_test,  scaler, t_idx, len(FEATURES))

mae  = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
mape = np.mean(np.abs((actual - pred) / actual)) * 100

print(f"\n📊 LSTM Results")
print(f"  MAE  : ${mae:.2f}")
print(f"  RMSE : ${rmse:.2f}")
print(f"  MAPE : {mape:.2f}%")

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(y=actual, name='Actual',
                          line=dict(color='#00d4ff', width=2)))
fig.add_trace(go.Scatter(y=pred,   name='Predicted',
                          line=dict(color='#ff6b6b', width=2, dash='dash')))
fig.update_layout(title=f'{TICKER} — LSTM Predictions vs Actual',
                   xaxis_title='Days', yaxis_title='Price (USD)',
                   template='plotly_dark', height=420)
fig.show()


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step 

📊 LSTM Results
  MAE  : $5.50
  RMSE : $7.22
  MAPE : 2.12%


In [ ]:
def run_prophet(df: pd.DataFrame, periods: int = 30) -> pd.DataFrame:
    prophet_df = df[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})

    m = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=0.05
    )
    m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    m.fit(prophet_df)

    future   = m.make_future_dataframe(periods=periods)
    forecast = m.predict(future)

    print(f"✅ Prophet forecast complete! Predicting next {periods} days.")
    return m, forecast

prophet_model, forecast = run_prophet(df, periods=30)

# Show last 10 predicted values
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10))


✅ Prophet forecast complete! Predicting next 30 days.
            ds        yhat  yhat_lower  yhat_upper
473 2026-05-20  272.432419  265.117015  278.948096
474 2026-05-21  272.287040  265.215534  279.041634
475 2026-05-22  272.515919  265.475628  279.981059
476 2026-05-23  270.391971  263.678531  277.364569
477 2026-05-24  271.355134  263.958291  279.115008
478 2026-05-25  275.677052  268.688483  282.607289
479 2026-05-26  276.983814  269.325490  284.300009
480 2026-05-27  278.033621  270.577528  285.889727
481 2026-05-28  277.755648  270.391162  284.730026
482 2026-05-29  277.666391  270.282524  284.705503


In [ ]:
# ── Plot 1: LSTM Predictions vs Actual ──────────────────────
fig1 = go.Figure()
fig1.add_trace(go.Scatter(y=actual, name='Actual Price',
                           mode='lines', line=dict(color='#00d4ff', width=2)))
fig1.add_trace(go.Scatter(y=pred,   name='LSTM Predicted',
                           mode='lines', line=dict(color='#ff6b6b', width=2, dash='dash')))
fig1.update_layout(title=f'{TICKER} — LSTM Predictions vs Actual',
                   xaxis_title='Days', yaxis_title='Price (USD)',
                   template='plotly_dark', height=450)
fig1.show()

# ── Plot 2: Training Loss ────────────────────────────────────
fig2 = go.Figure()
fig2.add_trace(go.Scatter(y=history.history['loss'],
                           name='Train Loss', line=dict(color='#00d4ff')))
fig2.add_trace(go.Scatter(y=history.history['val_loss'],
                           name='Val Loss',   line=dict(color='#ff6b6b')))
fig2.update_layout(title='LSTM Training Loss',
                   xaxis_title='Epoch', yaxis_title='Loss',
                   template='plotly_dark', height=350)
fig2.show()

# ── Plot 3: Sentiment Over Time ──────────────────────────────
fig3 = px.bar(df, x='Date', y='sentiment',
               color='sentiment',
               color_continuous_scale=['#ff6b6b', '#ffeaa7', '#00b894'],
               title=f'{TICKER} — Daily Sentiment Score',
               template='plotly_dark')
fig3.update_layout(height=350)
fig3.show()

print("✅ All charts rendered!")


✅ All charts rendered!


In [ ]:
# Write the Streamlit app to a file
streamlit_code = '''
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from datetime import datetime, timedelta

st.set_page_config(page_title="📈 Stock Sentiment Dashboard",
                   layout="wide", page_icon="📊")

st.markdown("""
    <style>
    .main { background-color: #0f0f1a; }
    h1, h2, h3 { color: #00d4ff; }
    .metric-container { background: #1a1a2e; border-radius: 10px; padding: 10px; }
    </style>
""", unsafe_allow_html=True)

st.title("📈 Stock Price Prediction & Sentiment Dashboard")
st.markdown("---")

# Sidebar
st.sidebar.header("⚙️ Configuration")
ticker  = st.sidebar.text_input("Stock Ticker", value="AAPL").upper()
period  = st.sidebar.selectbox("History Period", ["6mo", "1y", "2y", "5y"], index=1)
horizon = st.sidebar.slider("Forecast Days (Prophet)", 7, 90, 30)

@st.cache_data(ttl=3600)
def load_data(ticker, period):
    df = yf.Ticker(ticker).history(period=period)
    df.reset_index(inplace=True)
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
    return df[["Date", "Open", "High", "Low", "Close", "Volume"]]

@st.cache_data
def add_indicators(df):
    df = df.copy()
    df["MA_7"]  = df["Close"].rolling(7).mean()
    df["MA_21"] = df["Close"].rolling(21).mean()
    delta = df["Close"].diff()
    gain, loss = delta.clip(lower=0), -delta.clip(upper=0)
    rs = gain.rolling(14).mean() / loss.rolling(14).mean()
    df["RSI"] = 100 - (100 / (1 + rs))
    ema12 = df["Close"].ewm(span=12).mean()
    ema26 = df["Close"].ewm(span=26).mean()
    df["MACD"] = ema12 - ema26
    sma20 = df["Close"].rolling(20).mean()
    std20 = df["Close"].rolling(20).std()
    df["BB_Upper"] = sma20 + 2 * std20
    df["BB_Lower"] = sma20 - 2 * std20
    df["Daily_Return"] = df["Close"].pct_change()
    df.dropna(inplace=True)
    return df

def mock_sentiment(df):
    np.random.seed(42)
    scores = []
    for _, row in df.iterrows():
        base  = row.get("Daily_Return", 0)
        score = np.clip(base * 3 + np.random.normal(0, 0.2), -1, 1)
        scores.append({"Date": row["Date"], "sentiment": round(score, 4)})
    return pd.DataFrame(scores)

with st.spinner(f"Loading {ticker} data..."):
    try:
        raw  = load_data(ticker, period)
        df   = add_indicators(raw)
        sent = mock_sentiment(df)
        df   = pd.merge(df, sent, on="Date", how="left")
        df["sentiment"].fillna(0, inplace=True)
        info = yf.Ticker(ticker).info
    except Exception as e:
        st.error(f"Error loading data: {e}")
        st.stop()

# KPI Row
col1, col2, col3, col4 = st.columns(4)
latest       = df["Close"].iloc[-1]
prev         = df["Close"].iloc[-2]
change       = latest - prev
change_pct   = change / prev * 100
avg_sent     = df["sentiment"].tail(30).mean()
rsi_val      = df["RSI"].iloc[-1]

col1.metric("💰 Current Price", f"${latest:.2f}", f"{change_pct:+.2f}%")
col2.metric("📊 RSI (14)", f"{rsi_val:.1f}",
            "Overbought" if rsi_val > 70 else ("Oversold" if rsi_val < 30 else "Neutral"))
col3.metric("🗣️ Avg Sentiment (30d)", f"{avg_sent:.3f}",
            "Bullish 🟢" if avg_sent > 0 else "Bearish 🔴")
col4.metric("📈 30d Return",
            f"{((latest / df['Close'].iloc[-30]) - 1) * 100:.2f}%")

st.markdown("---")

# Price Chart with Indicators
tab1, tab2, tab3, tab4 = st.tabs(["📈 Price & Indicators", "🔮 Prophet Forecast",
                                    "🗣️ Sentiment", "📋 Raw Data"])

with tab1:
    fig = go.Figure()
    fig.add_trace(go.Candlestick(x=df["Date"], open=df["Open"],
                                  high=df["High"], low=df["Low"],
                                  close=df["Close"], name="OHLC"))
    fig.add_trace(go.Scatter(x=df["Date"], y=df["MA_7"],
                              line=dict(color="#00d4ff", width=1.5), name="MA 7"))
    fig.add_trace(go.Scatter(x=df["Date"], y=df["MA_21"],
                              line=dict(color="#a29bfe", width=1.5), name="MA 21"))
    fig.add_trace(go.Scatter(x=df["Date"], y=df["BB_Upper"],
                              line=dict(color="#fdcb6e", width=1, dash="dot"),
                              name="BB Upper"))
    fig.add_trace(go.Scatter(x=df["Date"], y=df["BB_Lower"],
                              fill="tonexty", line=dict(color="#fdcb6e", width=1, dash="dot"),
                              fillcolor="rgba(253,203,110,0.07)", name="BB Lower"))
    fig.update_layout(template="plotly_dark", height=500,
                       title=f"{ticker} Price Chart",
                       xaxis_rangeslider_visible=False)
    st.plotly_chart(fig, use_container_width=True)

    col_rsi, col_macd = st.columns(2)
    with col_rsi:
        fig_rsi = px.line(df, x="Date", y="RSI", title="RSI (14)",
                           template="plotly_dark", color_discrete_sequence=["#ff6b6b"])
        fig_rsi.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="Overbought")
        fig_rsi.add_hline(y=30, line_dash="dash", line_color="green", annotation_text="Oversold")
        st.plotly_chart(fig_rsi, use_container_width=True)
    with col_macd:
        fig_macd = px.line(df, x="Date", y="MACD", title="MACD",
                            template="plotly_dark", color_discrete_sequence=["#a29bfe"])
        st.plotly_chart(fig_macd, use_container_width=True)

with tab2:
    try:
        from prophet import Prophet
        prophet_df = df[["Date", "Close"]].rename(columns={"Date": "ds", "Close": "y"})
        m = Prophet(weekly_seasonality=True, yearly_seasonality=True,
                    changepoint_prior_scale=0.05)
        m.fit(prophet_df)
        future   = m.make_future_dataframe(periods=horizon)
        forecast = m.predict(future)

        fig_p = go.Figure()
        fig_p.add_trace(go.Scatter(x=prophet_df["ds"], y=prophet_df["y"],
                                    mode="lines", name="Historical",
                                    line=dict(color="#00d4ff")))
        fig_p.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat"],
                                    mode="lines", name="Forecast",
                                    line=dict(color="#a29bfe", dash="dash")))
        fig_p.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat_upper"],
                                    fill=None, mode="lines",
                                    line=dict(color="#6c5ce7", width=0), showlegend=False))
        fig_p.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat_lower"],
                                    fill="tonexty", mode="lines",
                                    line=dict(color="#6c5ce7", width=0),
                                    fillcolor="rgba(108,92,231,0.15)", name="CI Band"))
        fig_p.update_layout(template="plotly_dark", height=500,
                             title=f"{ticker} — {horizon}-Day Prophet Forecast")
        st.plotly_chart(fig_p, use_container_width=True)

        next_pred = forecast["yhat"].iloc[-1]
        ci_low    = forecast["yhat_lower"].iloc[-1]
        ci_high   = forecast["yhat_upper"].iloc[-1]
        st.info(f"📅 **{horizon}-day forecast**: ${next_pred:.2f}  "
                f"(Range: ${ci_low:.2f} – ${ci_high:.2f})")
    except Exception as e:
        st.error(f"Prophet error: {e}")

with tab3:
    fig_s = px.bar(df, x="Date", y="sentiment",
                    color="sentiment",
                    color_continuous_scale=["#ff6b6b", "#ffeaa7", "#00b894"],
                    title="Daily Sentiment Score", template="plotly_dark")
    st.plotly_chart(fig_s, use_container_width=True)

    rolling_sent = df.set_index("Date")["sentiment"].rolling(7).mean()
    st.line_chart(rolling_sent, use_container_width=True)

with tab4:
    st.dataframe(df.sort_values("Date", ascending=False).head(50),
                  use_container_width=True)

st.markdown("---")
st.caption("Built with ❤️ using Yahoo Finance · Prophet · VADER · Streamlit")
'''

with open("app.py", "w") as f:
    f.write(streamlit_code)

print("✅ app.py written!")


✅ app.py written!


In [ ]:
import subprocess, time, threading

# Kill any existing Streamlit
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(1)

# Start Streamlit in background
proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501",
     "--server.headless", "true", "--server.enableCORS", "false"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)

# Create public tunnel via pyngrok
from pyngrok import ngrok, conf

# If you have an ngrok auth token, set it here:
ngrok.set_auth_token("2zLtjxQ6QNAfhFCaikrdT9WuAwS_5YCNXVkeYfJqQfFdUM6ZN")

public_url = ngrok.connect(8501)
print("=" * 60)
print(f"🚀 Streamlit Dashboard is LIVE at:")
print(f"   {public_url}")
print("=" * 60)
print("📌 Open the link above in your browser!")
print("   (Ctrl+C in this cell to stop the server)")


🚀 Streamlit Dashboard is LIVE at:
   NgrokTunnel: "https://6680-34-126-188-189.ngrok-free.app" -> "http://localhost:8501"
📌 Open the link above in your browser!
   (Ctrl+C in this cell to stop the server)


In [ ]:
from pyngrok import ngrok
ngrok.disconnect(public_url)
ngrok.kill()
proc.terminate()
print("✅ Server stopped.")
